# WP6 — Conformal sufficiency layer (you implement)

**This is your work.** The conformal layer is the paper's central methodological contribution and you own its implementation. Each section below has a short math recipe (with the relevant equation from Angelopoulos & Bates 2021), a scaffold with hints, and an assertion cell that catches obvious bugs.

**Before you start.**

1. Make sure you've read Angelopoulos & Bates §1–3 and that you've written the tutorial summary from Milestone 1.
2. The settings:
   - Miscoverage rate **α = 0.10** (90 % nominal coverage)
   - Calibration recipe: **per cumulative-k** (recipe b in plan §2.4) — for a held-out participant's prediction at night k, the calibration set is *the other 41 participants' nonconformity scores at the same k*. Not pooled across k. Not final-k only.
   - Non-conformity score: **APS for binary** — `s(x, y) = 1 - p̂(y | x)` (Romano et al. 2020 §3, Angelopoulos & Bates §3.2).
3. If you get genuinely stuck after ~30 minutes on a section, look at `_internal/notebooks/04_wp6_reference.ipynb` (supervisor-only) for one possible solution. Use it as a check after you have your own attempt, not as a starting point.

**What you will build, end-to-end:**

1. Non-conformity scores for every (participant, night).
2. The conformal quantile q̂ per cumulative-k under LOSO.
3. Prediction sets `C_α(x)` with empty-set fallback.
4. The sufficiency threshold τᵢ per participant.
5. The decision file (`decisions/covariate_conditional.parquet`) and the τᵢ table.
6. The |C_α|-vs-k plot (provided — your focus is the algorithm).

**At the end**, a self-check cell verifies your outputs match expected summary statistics. If it passes, you have implemented split conformal prediction with APS correctly.

## 0. Setup (provided)

In [ ]:
import sys, os
from pathlib import Path

# Walk up from cwd to find the repo root (works from any directory)
_p = Path().resolve()
while not (_p / 'utils' / 'dataset.py').is_file() and _p != _p.parent:
    _p = _p.parent
REPO_ROOT = _p
sys.path.insert(0, str(REPO_ROOT))
os.chdir(REPO_ROOT)

import numpy as np
import pandas as pd
from utils.preview import peek

try:
    from google.colab import data_table
    data_table.enable_dataframe_formatter()
except ImportError:
    pass

DATA = Path('synthetic/v1')
(DATA / 'decisions').mkdir(parents=True, exist_ok=True)
ALPHA = 0.10
print(f'Pipeline root: {DATA}')
print(f'Miscoverage rate α = {ALPHA}')

## 1. Load inputs (provided)

Three files: probabilities, true labels, covariates. Read `docs/data_contract_v1.md` if any column name is unfamiliar.

In [ ]:
prob = pd.read_parquet(DATA / 'probability_table.parquet')
lbls = pd.read_parquet(DATA / 'labels.parquet')
cov  = pd.read_parquet(DATA / 'covariates.parquet')

joined = prob.merge(lbls, on=['participant_id', 'night_index'])
pids = sorted(prob['participant_id'].unique())

print(f'{len(joined):,} participant-night rows; {len(pids)} participants')
joined.head(3)

## 2. Non-conformity scores — APS for binary

**Math (Romano et al. 2020 §3; Angelopoulos & Bates eq. 6).** For each example with features $x$ and *true* label $y$:

$$s(x, y) = 1 - \hat{p}(y \mid x)$$

Confident-and-correct predictions get small scores; unconfident or wrong predictions get large scores.

**For our binary case:**
- If `binary_label == 'post'`, the true class is post, so $s = 1 - p_{\text{post}}$.
- If `binary_label == 'pre'`, the true class is pre, so $s = 1 - (1 - p_{\text{post}}) = p_{\text{post}}$.

**Your task:** add a `nonconformity` column to `joined`.

In [ ]:
# YOUR CODE HERE — compute nonconformity scores per (participant, night)
#
# Hint: np.where(condition, value_if_true, value_if_false) is a one-liner
# that handles both classes simultaneously.

raise NotImplementedError('Section 2: compute joined["nonconformity"]')

In [ ]:
# Self-check — these assertions catch the most common mistakes.
assert 'nonconformity' in joined.columns, 'add a nonconformity column to joined'
s = joined['nonconformity']
assert (s >= 0).all() and (s <= 1).all(), 'scores must lie in [0, 1]'
# When the prediction is confident AND correct, score should be close to 0.
# When the prediction is confidently WRONG, score should be close to 1.
n_post = (joined['binary_label'] == 'post').sum()
post_low_score = ((joined['binary_label'] == 'post') & (joined['nonconformity'] < 0.1)).sum()
print(f'post-labeled rows: {n_post:,}, of which {post_low_score:,} have nonconformity < 0.1 (confident & correct)')
print(f'mean nonconformity: {s.mean():.3f} (typical range 0.10–0.35 on synthetic data)')
print('OK')

## 3. LOSO conformal quantile — per cumulative-k

**Math (Angelopoulos & Bates eq. 4).** Given calibration scores $\{s_1, \ldots, s_n\}$, the conformal quantile at miscoverage α is:

$$\hat{q} = \text{Quantile}_{\lceil (n+1)(1-\alpha) \rceil / n}\bigl(\{s_1, \ldots, s_n\}\bigr)$$

**Recipe (b) — per-k calibration (plan §2.4 "open items"):** for a held-out participant's prediction at cumulative night $k$, the calibration set is *the other 41 participants' nonconformity scores at exactly the same $k$*. **Not pooled across k.** Not final-k only.

**Why per-k?** The sufficiency story is about uncertainty *at a specific cumulative night*. Pooling across $k$ mixes the early-uncertain regime with the late-confident regime and produces inverted trajectories.

**Your task:** build `q_hat_table[(pid, k)] -> q̂_k`.

In [ ]:
# YOUR CODE HERE — for each held-out participant pid and each cumulative_k k,
# compute q_hat_table[(pid, k)] using only the OTHER 41 participants' scores at the same k.
#
# Hints:
#   1. Outer loop over pids; inner loop (or groupby) over cumulative_k.
#   2. To get the calibration scores: joined[(joined.participant_id != pid) & (joined.cumulative_k == k)]['nonconformity']
#   3. The level for np.quantile is min(np.ceil((n + 1) * (1 - ALPHA)) / n, 1.0).
#   4. Pass method='higher' to np.quantile so you take the score at or above the level
#      (rather than interpolating between two scores — interpolation breaks the conformal guarantee).

q_hat_table = {}
raise NotImplementedError('Section 3: populate q_hat_table[(pid, k)]')

In [ ]:
# Self-check
assert q_hat_table, 'q_hat_table is empty — your loop did not populate it'
vals = list(q_hat_table.values())
assert all(0 <= v <= 1 for v in vals), 'every q̂ should lie in [0, 1]'
n_pids = len(set(k[0] for k in q_hat_table))
ks = sorted(set(k[1] for k in q_hat_table))
print(f'computed q̂ for {len(q_hat_table):,} (pid, k) pairs')
print(f'  participants: {n_pids} (expected {len(pids)})')
print(f'  cumulative_k range: {min(ks)}–{max(ks)}')
print(f'  q̂ range: [{min(vals):.3f}, {max(vals):.3f}], median {np.median(vals):.3f}')
print('OK')

## 4. Construct prediction sets

**Math (Angelopoulos & Bates eq. 5; specialized to binary).** A class $y$ is included in the prediction set iff its predicted probability is at least $1 - \hat{q}_k$:

$$C_\alpha(x) = \{ y \,:\, \hat{p}(y \mid x) \geq 1 - \hat{q}_k \}$$

**Empty-set fallback.** Basic APS can produce an empty set when both classes have $\hat{p} < 1 - \hat{q}$. Romano et al. 2020 §3 introduces a non-empty variant. For our binary case, the simplest equivalent fix: **if both classes would be excluded, include the argmax class.** This guarantees $|C_\alpha| \geq 1$.

**Your task:** for each row in `joined`, compute `contains_pre`, `contains_post`, and `set_size = contains_pre + contains_post`. Save into a list of dicts that becomes `prediction_sets.parquet`.

**Schema of `prediction_sets.parquet`** (from `docs/pipeline_contract_v1.md` §7):

```
participant_id (str), night_index (int32), cumulative_k (int32),
alpha (float32), contains_pre (bool), contains_post (bool), set_size (int8),
nonconformity_score_pre (float32), nonconformity_score_post (float32),
conformal_quantile (float32), mondrian_stratum (str), is_synthetic (bool)
```

Use `mondrian_stratum = 'global'` for now (Mondrian is journal-version work).

In [ ]:
# YOUR CODE HERE — for every (participant, night), construct the prediction set.
#
# Hints:
#   - Iterate participants → rows. For each row, look up q̂ from q_hat_table[(pid, k)].
#   - p_post = row.p_post_ovulatory; p_pre = 1 - p_post.
#   - Apply the rule. Apply the empty-set fallback (include argmax) when needed.
#   - Append a dict with the schema above to `rows`.

rows = []
raise NotImplementedError('Section 4: build rows for prediction_sets')

# After your loop:
# ps = pd.DataFrame(rows).astype({
#     'participant_id': 'string', 'night_index': 'int32', 'cumulative_k': 'int32',
#     'alpha': 'float32', 'contains_pre': 'bool', 'contains_post': 'bool',
#     'set_size': 'int8', 'nonconformity_score_pre': 'float32',
#     'nonconformity_score_post': 'float32', 'conformal_quantile': 'float32',
#     'mondrian_stratum': 'string', 'is_synthetic': 'bool',
# })
# ps.to_parquet(DATA / 'prediction_sets.parquet', index=False)

In [ ]:
# Self-check
ps = pd.read_parquet(DATA / 'prediction_sets.parquet')
assert len(ps) == len(joined), f'prediction_sets has {len(ps)} rows, expected {len(joined)}'
assert ps['set_size'].isin([1, 2]).all(), 'set_size must be 1 or 2 (empty-set fallback should rule out 0)'
ss_dist = ps['set_size'].value_counts().sort_index().to_dict()
print(f'set_size distribution: {ss_dist}')
frac_singleton = (ps['set_size'] == 1).mean()
print(f'fraction with |C|=1: {frac_singleton:.2%} (typical 65–85% on synthetic)')
print('OK')

## 5. Sufficiency threshold τᵢ + decision file

**Definition (plan §2.3).** $\tau_i$ is the smallest $k$ such that $|C_\alpha(x_i^{1:k})| = 1$ for at least 2 consecutive observation windows. Non-converger: $\tau_i$ is null.

**Decision rule (pipeline contract §5, `covariate_conditional`):**
- `decision = 'predict'` from $\tau_i$ onward — predicted_label is whichever class is in the singleton.
- `decision = 'defer'` before $\tau_i$ (when $|C| = 2$).
- `decision = 'collect_more'` is reserved for a later WP2 quality-flag column; emit `defer` for now.

**Your task:** for each participant, find τᵢ, then emit one decision row per night with the correct field values.

In [ ]:
# YOUR CODE HERE — compute τᵢ per participant, emit dec_rows and tau_rows.
#
# Hints:
#   - For τᵢ: iterate nights of one participant (sort by night_index). Find first i where
#     set_size[i] == 1 AND set_size[i+1] == 1. τᵢ = night_index[i]. None if no such i.
#   - For decisions: for nights at or after τᵢ, predicted_label is 'pre' or 'post' depending
#     on which class is in the singleton (contains_post=True → 'post'; contains_pre=True → 'pre').
#   - For nights before τᵢ: decision='defer', predicted_label=None.
#   - For non-convergers: ALL nights are 'defer'.

dec_rows, tau_rows = [], []
raise NotImplementedError('Section 5: compute tau, build dec_rows and tau_rows')

# After your loop:
# decisions = pd.DataFrame(dec_rows).astype({
#     'participant_id': 'string', 'night_index': 'int32', 'cumulative_k': 'int32',
#     'strategy_name': 'string', 'decision': 'string',
#     'predicted_label': 'string', 'prediction_set_size': 'Int8',
#     'mondrian_stratum': 'string', 'is_synthetic': 'bool',
# })
# decisions.to_parquet(DATA / 'decisions/covariate_conditional.parquet', index=False)

In [ ]:
# Self-check
decisions = pd.read_parquet(DATA / 'decisions/covariate_conditional.parquet')
assert len(decisions) == len(joined), f'decisions has {len(decisions)} rows, expected {len(joined)}'
assert decisions['decision'].isin(['predict', 'defer']).all(), 'decision must be predict or defer in v1'
predicts = (decisions['decision'] == 'predict').sum()
defers = (decisions['decision'] == 'defer').sum()
print(f'decisions: {predicts:,} predict, {defers:,} defer')
n_converged = sum(1 for r in tau_rows if r['convergence_status'] == 'converged')
print(f'converged: {n_converged}/{len(pids)} participants')

## 6. Append τᵢ rows to `tau_per_strategy.parquet`

Pipeline contract §6: this is the one shared file appended to by both WP5 (its 6 baselines) and WP6 (covariate_conditional). Read existing rows, drop any prior `covariate_conditional` rows, append yours, write back.

**Your task:** the schema is on the contract; the logic is mechanical.

In [ ]:
# YOUR CODE HERE — append tau_rows to tau_per_strategy.parquet.
#
# Schema (pipeline contract §6):
#   participant_id (string), strategy_name (string),
#   tau_i (Int32, nullable for non-convergers), convergence_status (string), is_synthetic (bool)
#
# Hint: read existing file if it exists, filter out rows where strategy_name == 'covariate_conditional'
# (you might be re-running this notebook), then concat and write back.

raise NotImplementedError('Section 6: append covariate_conditional rows to tau_per_strategy.parquet')

## 7. Milestone-3 plot (provided)

Your focus is the algorithm; the plot is mechanical. The cell below reads `prediction_sets.parquet` (which you just wrote) and produces the |C_α|-vs-k plot for ~10 representative participants. **Save this plot and send it to supervisor — it is the Milestone 3 gate.**

In [ ]:
import matplotlib.pyplot as plt
ps = pd.read_parquet(DATA / 'prediction_sets.parquet')
ss = cov.set_index('participant_id')['signal_completeness']
sample_pids = ss.sort_values().iloc[[0, 7, 14, 21, 27, 34, 41]].index.tolist()

fig, ax = plt.subplots(figsize=(8, 4.5))
for pid in sample_pids:
    g = ps[ps['participant_id'] == pid].sort_values('night_index')
    ax.plot(g['night_index'], g['set_size'], alpha=0.8, label=f'{pid} (sc={ss[pid]:.2f})')
ax.axhline(1, color='black', linestyle='--', linewidth=0.7, label='|C_α|=1 (sufficiency)')
for k in [3, 5, 7]:
    ax.axvline(k, color='grey', linestyle=':', linewidth=0.7)
ax.set_xlabel('cumulative valid nights (k)')
ax.set_ylabel('|C_α|')
ax.set_title(f'Individual sufficiency trajectories (synthetic, α={ALPHA})')
ax.legend(loc='upper right', fontsize=7)
ax.set_yticks([1, 2])
Path('docs/figures').mkdir(parents=True, exist_ok=True)
out = Path('docs/figures/student3_m3_sufficiency_curves.png')
fig.tight_layout(); fig.savefig(out, dpi=150); plt.show()
print(f'saved {out}')

## 8. Self-check against expected statistics

If everything above is correct, the assertions below pass. Each one targets a specific class of bug.

If one fails, scroll up and re-read the math for that section. The reference solution at `_internal/notebooks/04_wp6_reference.ipynb` is one possible solution; your code can differ stylistically as long as the statistics match.

In [ ]:
ps = pd.read_parquet(DATA / 'prediction_sets.parquet')
tau = pd.read_parquet(DATA / 'tau_per_strategy.parquet')
tau_cc = tau[tau['strategy_name'] == 'covariate_conditional']
decisions = pd.read_parquet(DATA / 'decisions/covariate_conditional.parquet')

# Expected statistics computed from the reference solution on synthetic seed=42.
# Your numbers should match within a small tolerance.
checks = [
    ('total prediction-set rows',  len(ps),              1352, 0),
    ('rows with |C|=1',            (ps['set_size']==1).sum(), 956, 30),
    ('rows with |C|=2',            (ps['set_size']==2).sum(), 396, 30),
    ('rows with |C|=0 (must be 0)', (ps['set_size']==0).sum(), 0,  0),
    ('participants',                tau_cc['participant_id'].nunique(), 42, 0),
    ('total decision rows',        len(decisions),       1352, 0),
    ('decisions=predict',          (decisions['decision']=='predict').sum(), 1352, 200),
    ('mean p_post_ovulatory in calibration', joined['p_post_ovulatory'].mean(), 0.55, 0.05),
]
ok = True
for label, value, expected, tol in checks:
    diff = abs(value - expected)
    status = 'OK ' if diff <= tol else 'FAIL'
    if status == 'FAIL':
        ok = False
    print(f'  {status}  {label:<40s}  got={value:<10}  expected={expected} (±{tol})')
print()
print('Self-check passed.' if ok else 'Self-check FAILED — re-read the math for the failing section.')

## What you've built

Reading top to bottom, you implemented:

1. **APS non-conformity scores** — the calibration target.
2. **LOSO with per-cumulative-k calibration (recipe b)** — the right pooling for a sufficiency story.
3. **Conformal prediction sets with empty-set fallback** — the Romano et al. 2020 nonempty variant for binary.
4. **Sufficiency threshold τᵢ** — first 2 consecutive singletons, plan §2.3.
5. **Decision policy** — predict / defer per pipeline contract §5.

This is the WP6 core method. The conformal sufficiency layer is now your code, run against your data, with your output files. Methodology section in your conference paper writes itself from the equations and assertions above.

## Next (Milestone 4 and onward)

- **Real data.** Re-run this notebook against `probability_table.parquet` produced by the WP5 student (or your stand-in classifier — see notatka day 4).
- **Mondrian conformal.** Compute per-stratum q̂ instead of one global per-k value. Strata: high/low signal_completeness × regular/irregular cycles. Add `mondrian_stratum` column accordingly.
- **Regression meta-model.** Fit a simple regression `τᵢ ~ covariates` to identify which covariate predicts convergence speed.
- **`collect_more` branch.** Activate once `valid_nights.quality_flag` is populated.